# Chapter 10 — Causal Bayes Nets and the Do-Operator

Companion notebook for the [probintro textbook](https://josephausterweil.github.io/probintro/intro2/10_causal_bayes_nets/).

See the difference between **observing** yellow teeth and **setting** them (the do-operator) in the smoking / teeth / cancer network.

## Setup

In [ ]:
!pip install genjax -q
print('✅ ready')

## Two models: observe vs. intervene

The only difference is that the intervened model *deletes* the teeth node — graph surgery.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
from genjax import gen, flip, ChoiceMap

P_SMOKE = 0.3
P_YELLOW_SMOKE, P_YELLOW_NOSMOKE = 0.8, 0.2
P_CANCER_SMOKE, P_CANCER_NOSMOKE = 0.15, 0.01

@gen
def observational():
    smoking = flip(P_SMOKE) @ "smoking"
    teeth = flip(jnp.where(smoking, P_YELLOW_SMOKE, P_YELLOW_NOSMOKE)) @ "teeth"
    cancer = flip(jnp.where(smoking, P_CANCER_SMOKE, P_CANCER_NOSMOKE)) @ "cancer"
    return cancer

@gen
def intervened_yellow():
    # do(teeth = yellow): the smoking -> teeth arrow is cut; teeth is left out.
    smoking = flip(P_SMOKE) @ "smoking"
    cancer = flip(jnp.where(smoking, P_CANCER_SMOKE, P_CANCER_NOSMOKE)) @ "cancer"
    return cancer

## See vs. do

In [ ]:
N = 60000
obs = ChoiceMap.d({"teeth": 1})
keys = jr.split(jr.key(0), N)
def observe_one(k):
    trace, log_weight = observational.generate(k, obs, ())
    return trace.get_choices()["cancer"].astype(float), log_weight
cancer_obs, log_w = jax.vmap(observe_one)(keys)
w = jnp.exp(log_w - jnp.max(log_w)); w = w / jnp.sum(w)
p_see = jnp.sum(cancer_obs * w)

cancer_do = jax.vmap(lambda k: intervened_yellow.simulate(k, ()).get_retval())(keys)
p_do = jnp.mean(cancer_do.astype(float))

print(f"P(cancer | teeth = yellow)      = {float(p_see):.3f}   (observe — confounded)")
print(f"P(cancer | do(teeth = yellow))  = {float(p_do):.3f}   (intervene — base rate)")

## Your turn

1. Build an `intervened_nosmoking` model for $do(\text{smoking} = \text{false})$ (a smoking ban) and estimate $P(\text{cancer} \mid do(\text{no smoking}))$.
2. Compare it to the *observational* $P(\text{cancer} \mid \text{no smoking})$. Are they equal? Why? (Smoking is a root — it has no parents to cut.)
3. Change the CPT so smoking is rarer (say `P_SMOKE = 0.1`). Does the see/do gap for teeth grow or shrink?